In [ ]:
from typing import List, Union, Dict, Any, Optional
from PIL import Image
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from tqdm import tqdm
import json
import re,os
import numpy as np

class Qwen2VL: 
    def __init__(
        self,
        model_path: str = "",
        tensor_parallel_size: int = 1,
        gpu_memory_utilization: float = 0.99,
        enforce_eager: bool = False,
        max_model_len: int = 10240,
        limit_mm_per_prompt: Dict[str, int] = {"image": 1, "video": 0},
        min_pixels: Optional[int] = None,
        max_pixels: Optional[int] = None,
        max_num_seqs: Optional[int] = None,
    ):
        """
        Initialize the Qwen2VL inference class
        
        Args:
            model_path: Model path
            tensor_parallel_size: Number of tensor parallel partitions
            gpu_memory_utilization: GPU memory utilization rate
            max_model_len: Maximum sequence length
            max_num_seqs: Maximum number of sequences
            min_pixels: Minimum number of pixels
            max_pixels: Maximum number of pixels
        """
        kwargs = {
            "model": model_path,
            "tensor_parallel_size": tensor_parallel_size,
            "gpu_memory_utilization": gpu_memory_utilization,
            "enforce_eager": enforce_eager,
            "max_model_len": max_model_len,
            "limit_mm_per_prompt": limit_mm_per_prompt,
        }
        if max_num_seqs:
            kwargs["max_num_seqs"] = max_num_seqs
        if min_pixels and max_pixels:
            kwargs["mm_processor_kwargs"] = {
                "min_pixels": min_pixels if min_pixels else 32 * 28 * 28,
                "max_pixels": max_pixels,
            }

        self.llm = LLM(**kwargs)
        self.system_prompt = "You are a helpful assistant."
        
        
    def _format_prompt(self, messages: List[Dict[str, Any]], images: List[Image.Image]) -> str:
        """
        Format conversation history, handling custom image positions
        
        Args:
            messages: Conversation history
            images: List of images
        """
        prompt = ""
        if messages[0]["role"] != "system":
            prompt = f"<|im_start|>system\n{self.system_prompt}<|im_end|>\n"
        
        image_idx = 0
        
        for message in messages:
            role = message["role"]
            content = message["content"]
            if role == "system":
                prompt += f"<|im_start|>system\n{content}<|im_end|>\n"
            elif role in ["user", "human"]:
                # Calculate required number of images
                required_images = content.count("<image>")
                if image_idx + required_images > len(images):
                    raise ValueError(f"Not enough images provided. Required: {image_idx + required_images}, Provided: {len(images)}")
                
                # Replace all <image> tags
                for _ in range(required_images):
                    content = content.replace("<image>", "<|vision_start|><|image_pad|><|vision_end|>", 1)
                    image_idx += 1
                    
                prompt += f"<|im_start|>user\n{content}<|im_end|>\n"
            elif role in ["assistant", "gpt"]:
                prompt += f"<|im_start|>assistant\n{content}<|im_end|>\n"
                
        prompt += "<|im_start|>assistant\n"
        
        # Verify that all images were used
        if image_idx < len(images):
            raise ValueError(f"Too many images provided. Used: {image_idx}, Provided: {len(images)}")
            
        return prompt

    def chat(
        self,
        messages: Union[List[Dict[str, Any]], List[List[Dict[str, Any]]]],
        images: Optional[Union[Image.Image, List[Image.Image], List[List[Image.Image]]]] = None,
        temperature: float = 0.,
        max_tokens: int = 2048,
        top_p: float = 0.95,
        **kwargs
    ) -> Union[str, List[str]]:

        # Determine if this is a batch request
        is_batch = isinstance(messages[0], list)
        if not is_batch:
            messages = [messages]
            images = [images] if isinstance(images, Image.Image) else ([images] if images is not None else [None])
            
        # Prepare inputs
        inputs = []
        assert len(messages) == len(images)
        for msg, img_list in zip(messages, images):
            if img_list is not None:
                img_list = [img_list] if isinstance(img_list, Image.Image) else img_list
                prompt = self._format_prompt(msg, img_list)
                
                # Build multimodal data dictionary
                input_data = {
                    "prompt": prompt,
                    "multi_modal_data": {
                        "image": img_list
                    }
                }
            else:
                prompt = self._format_prompt(msg, [])
                input_data = {
                    "prompt": prompt,
                    "multi_modal_data": {}
                }
                
            inputs.append(input_data)
            
        # Set sampling parameters
        sampling_params = SamplingParams(
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            **kwargs
        )
        
        # Execute inference
        outputs = self.llm.generate(inputs, sampling_params=sampling_params)
        responses = [output.outputs[0].text for output in outputs]
        
        return responses[0] if not is_batch else responses

model_path = "./LAMO-3B"
tensor_parallel_size = 4
gpu_memory_utilization = 0.94
max_model_len = 13400
min_pixels = 32*28*28
max_pixels = 10240*28*28
# for ScreenSport-pro
# max_model_len = 38000
# max_pixels = 55000*28*28
# temperature= 0.05

llm = Qwen2VL(model_path, tensor_parallel_size, gpu_memory_utilization, max_model_len=max_model_len, min_pixels=min_pixels, max_pixels=max_pixels)


In [ ]:
from PIL import Image
WEB_PC_ACTION_SPACE = """
{"action": pyautogui.click(x=x1, y=y1), "description":"click on the element at the coordinate point (x1, y1)"}
{"action": pyautogui.doubleClick(x=x1, y=y1), "description":"double-click the (x1, y1) position on the screen"}
{"action": pyautogui.rightClick(x=x1, y=y1), "description":"right-click the (x1, y1) position on the screen"}
{"action": pyautogui.moveTo(x=x1, y=y1), "description":"move the cursor to (x1, y1)"}
{"action": pyautogui.dragTo(x=x1, y=y1), "description":"drag the cursor to (x1, y1)"}
{"action": pyautogui.press(keys=['key']), "description":"press a key on the keyboard"}
{"action": pyautogui.hotkey(keys=['key1', 'key2', ...]), "description":"press keyboard shortcut combinations"}
{"action": pyautogui.write(message='text'), "description":"type a string of text"}
{"action": pyautogui.scroll('up/down/left/right'), "description":"scroll the mouse wheel in a specific direction"}
{"action": pyautogui.wait(), "description":"waiting for loading"}
{"action": pyautogui.terminate(status='success'), "description":"goal has been achieved"}"""

INSTRUCTION_EXECUTOR = """You are given an instruction and a screenshot. You need to perform one or more actions to align the instruction.
Here are the tools you can use: {WEB_PC_ACTION_SPACE}

Instruction: {ACTION}

Please keep the following output format: 
<action>a brief description of the current action</action>
<tool_call>determine the execution tool/tools based on the current action</tool_call>"""

img = Image.open("./demo.png")

action = 'Click the input box with the current value "80" under "Editor: Word Wrap Column".'
# action = 'Click the "ok" button.'

# ELEMENT_DESCRIPTION = "the close icon"
# INSTRUCTION_GROUNDING = """<image>Locate the element on the screen with the function or description: {ELEMENT_DESCRIPTION}.
# Keep the following output format: {"point_2d": [x, y], "label": "re-describe the element to help you grounding"}."""

prompt = f'''<image>{INSTRUCTION_EXECUTOR.format(ACTION=action, WEB_PC_ACTION_SPACE=WEB_PC_ACTION_SPACE)}'''
prompt = prompt.replace("(action)", action)
msg =  [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}]
pred_res = llm.chat(messages=msg, images=[img], temperature= 0.05, max_tokens = 2048, top_p = 0.95)
print(pred_res)